# DYSIB Pendulum: Reproduce Figures

This notebook reproduces every figure in the paper from the precomputed analysis files already included in this repository under `figure_data/`.
**Nothing in `figure_data/` needs to be regenerated to run this notebook.** If you just want the paper figures, open the notebook and run it as-is.
**No training or GPU required.** The notebook runs with only `numpy`, `scipy`, `pandas`, `matplotlib`, and `cmocean`.

`precompute_figure_data.py` is only needed if you have trained new models or otherwise want to replace the shipped analysis files with outputs derived from new checkpoints.
```bash
python precompute_figure_data.py       # rebuild figure_data/ for new training outputs
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
import cmocean
from scipy.stats import norm

# ── global matplotlib defaults ────────────────────────────────────────────
mpl.rcParams.update({
    "figure.figsize": (5, 5),
    "figure.dpi":     150,
    "font.size":      14,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
})
plt.rcParams["font.family"]      = "serif"
plt.rcParams["font.serif"]       = ["STIXGeneral"]
plt.rcParams["mathtext.fontset"] = "cm"

# ── paths + save config ───────────────────────────────────────────────────
FIGDATA  = Path("figure_data")
SAVE_DIR = Path("Figures");  SAVE_DIR.mkdir(exist_ok=True)
SAVE_DPI = 600   # increase to 1000 for publication quality

# ── load sweep summary table (571 unique runs) ────────────────────────────
df_sweep = pd.read_csv(FIGDATA / "sweep_metrics.csv")
print(f"Loaded {len(df_sweep)} runs; cols: {list(df_sweep.columns)}")

# ── colormaps (match the paper) ───────────────────────────────────────────
CMAP_THETA = cmocean.cm.phase
CMAP_OMEGA = plt.cm.RdBu
CMAP_KE    = plt.cm.viridis
CMAP_PE    = plt.cm.PiYG
CMAP_TE    = plt.cm.inferno

# ── unit conversions (match paper: degrees, deg/s) ────────────────────────
_MI_SCALE     = 1.0 / np.log(2)    # nats → bits
_OMEGA_SCALE  = 180.0 / np.pi      # rad/s → deg/s

# ── plot-mode switch (matches paper_figures_pub.ipynb) ────────────────────
#   "mean_std"        : error-bars of mean ± std across repetitions (+ fill)
#   "panel_best"      : scatter all runs; draw line through the best run per x
#                       in EACH panel (best = max final_test_mi for MI, min RMSE for probers)
#   "shared_highlight": scatter all runs; draw line through the best run per x
#                       using a single shared selector (HIGHLIGHT_SELECTOR)
PLOT_MODE          = "panel_best"
HIGHLIGHT_SELECTOR = "max_test_mi"

SHARED_SELECTORS = {
    "max_test_mi":    ("final_test_mi",   "max"),
    "min_theta_test": ("theta_test_deg",  "min"),
    "min_omega_test": ("omega_test_rmse", "min"),
}

# scatter cosmetics for panel_best / shared_highlight modes
SCATTER_ALPHA        = 0.05
SCATTER_SIZE         = 28
HIGHLIGHT_SIZE       = 72
HIGHLIGHT_EDGEWIDTH  = 0.9

print("Setup done.")


### Shared helpers: aggregate sweep metrics → mean ± std
Used by all three sweep figures.

In [ ]:
def _agg(df, x_col, metric_col):
    """Group by x_col, return DataFrame (x, mean, std)."""
    g = df.groupby(x_col)[metric_col].agg(['mean', 'std']).reset_index().sort_values(x_col)
    g['std'] = g['std'].fillna(0.0)
    return g

PANEL_SPECS = [
    {"key": "mi",    "test": "final_test_mi",    "scale": _MI_SCALE,    "ylabel": "MI (bits)",                                       "title": "Information",             "panel_best": ("final_test_mi",   "max")},
    {"key": "theta", "test": "theta_test_deg",   "scale": 1.0,          "ylabel": r"$\theta$ error ($^\circ$)",                       "title": "Angle Prober",            "panel_best": ("theta_test_deg",  "min")},
    {"key": "omega", "test": "omega_test_rmse",  "scale": _OMEGA_SCALE, "ylabel": r"$\omega$ error ($^\circ$/s)",                     "title": "Angular Velocity Prober", "panel_best": ("omega_test_rmse", "min")},
]

def _best_per_x(df, x_col, metric_col, reducer):
    """For each unique x, return row with min/max metric_col."""
    best_idx = []
    for _, bucket in df.groupby(x_col, sort=True):
        s = bucket[metric_col].replace([np.inf, -np.inf], np.nan).dropna()
        if s.empty:
            continue
        best_idx.append(s.idxmin() if reducer == 'min' else s.idxmax())
    return df.loc[best_idx].sort_values(x_col) if best_idx else df.iloc[0:0]

def plot_sweep_panel(ax, df, spec, x_col, *, mode=None, fill_std=True,
                     color='steelblue'):
    """Plot one panel in the chosen mode. Defaults to the global PLOT_MODE."""
    mode = mode or PLOT_MODE
    scale = spec['scale']

    if mode == 'mean_std':
        agg = _agg(df, x_col, spec['test'])
        ax.errorbar(agg[x_col], agg['mean'] * scale, yerr=agg['std'] * scale,
                    marker='o', lw=1.8, ms=5, capsize=3, color=color)
        if fill_std:
            ax.fill_between(agg[x_col],
                            (agg['mean'] - agg['std']) * scale,
                            (agg['mean'] + agg['std']) * scale,
                            color=color, alpha=0.15, linewidth=0)
        return

    # scatter every run (translucent)
    ax.scatter(df[x_col], df[spec['test']] * scale,
               s=SCATTER_SIZE, alpha=SCATTER_ALPHA, color=color, marker='o',
               linewidths=0)

    # pick selector
    if mode == 'panel_best':
        metric_col, reducer = spec['panel_best']
    elif mode == 'shared_highlight':
        metric_col, reducer = SHARED_SELECTORS[HIGHLIGHT_SELECTOR]
    else:
        raise ValueError(f"Unknown mode: {mode!r}")

    best = _best_per_x(df, x_col, metric_col, reducer)
    if best.empty:
        return
    ax.plot(best[x_col], best[spec['test']] * scale,
            color=color, lw=1.8, alpha=1.0)
    ax.scatter(best[x_col], best[spec['test']] * scale,
               s=HIGHLIGHT_SIZE, color=color, alpha=1.0, marker='o',
               edgecolors='black', linewidths=HIGHLIGHT_EDGEWIDTH, zorder=5)


## Figure: Sweep vs $n_F$ (kz=2, N=1000)

In [ ]:
# ── font sizes & cosmetics ────────────────────────────────────────────────
FS_TICK, FS_AXIS_LABEL, FS_TITLE, FS_PANEL_LABEL = 15, 20, 21, 24
GRID_ALPHA = 0.10

d = df_sweep[(df_sweep.samples_n == 1000) & (df_sweep.ndims == 2) & (df_sweep.num_frames <= 8)]

fig, ax_dict = plt.subplot_mosaic(
    [["mi", "theta"], ["mi", "omega"]],
    figsize=(11, 7), dpi=200,
    gridspec_kw=dict(width_ratios=[1.3, 1], hspace=0.45, wspace=0.38),
)
for spec in PANEL_SPECS:
    plot_sweep_panel(ax_dict[spec['key']], d, spec, 'num_frames')

ax_dict['mi'].set_xlim([0.8, 8.2])
for key, ax in ax_dict.items():
    ax.set_xlabel(r"Number of frames $n_F$", fontsize=FS_AXIS_LABEL)
    ax.tick_params(labelsize=FS_TICK)
    ax.grid(alpha=GRID_ALPHA, linewidth=0.5)
    spec = next(s for s in PANEL_SPECS if s['key'] == key)
    ax.set_ylabel(spec['ylabel'], fontsize=FS_AXIS_LABEL)
    ax.set_title(spec['title'], fontsize=FS_TITLE)

for ax, lbl in zip([ax_dict['mi'], ax_dict['theta'], ax_dict['omega']], 'ABC'):
    y = 1.05 if lbl == 'A' else 1.12
    ax.text(-0.05, y, lbl, transform=ax.transAxes, fontsize=FS_PANEL_LABEL)

plt.savefig(SAVE_DIR / 'Fig_Sweep_vs_nF.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## Figure: Sweep vs $k_z$ (nF=2, N=1000)

In [ ]:
FS_TICK, FS_AXIS_LABEL, FS_TITLE, FS_PANEL_LABEL = 15, 20, 21, 24
GRID_ALPHA = 0.10

d = df_sweep[(df_sweep.samples_n == 1000) & (df_sweep.num_frames == 2)]

fig, ax_dict = plt.subplot_mosaic(
    [["mi", "theta"], ["mi", "omega"]],
    figsize=(11, 7), dpi=200,
    gridspec_kw=dict(width_ratios=[1.3, 1], hspace=0.45, wspace=0.38),
)
for spec in PANEL_SPECS:
    plot_sweep_panel(ax_dict[spec['key']], d, spec, 'ndims', fill_std=True)
    ax_dict[spec['key']].set_xscale('log')

ax_dict['mi'].set_ylim([0, 10.2])
for key, ax in ax_dict.items():
    ax.set_xticks([1, 2, 4, 8, 16, 32])
    ax.set_xticklabels(['1', '2', '4', '8', '16', '32'])
    ax.set_xlabel(r"Latent dimension $k_z$", fontsize=FS_AXIS_LABEL)
    ax.tick_params(labelsize=FS_TICK)
    ax.grid(alpha=GRID_ALPHA, linewidth=0.5)
    spec = next(s for s in PANEL_SPECS if s['key'] == key)
    ax.set_ylabel(spec['ylabel'], fontsize=FS_AXIS_LABEL)
    ax.set_title(spec['title'], fontsize=FS_TITLE)

for ax, lbl in zip([ax_dict['mi'], ax_dict['theta'], ax_dict['omega']], 'ABC'):
    y = 1.05 if lbl == 'A' else 1.12
    ax.text(-0.05, y, lbl, transform=ax.transAxes, fontsize=FS_PANEL_LABEL)

plt.savefig(SAVE_DIR / 'Fig_Sweep_vs_kz.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## Figure: Sweep vs $N$ (kz=2, nF=2)

In [ ]:
# ── font sizes & cosmetics ────────────────────────────────────────────────
FS_TICK, FS_AXIS_LABEL, FS_TITLE, FS_PANEL_LABEL = 15, 20, 21, 24
GRID_ALPHA = 0.10

d = df_sweep[(df_sweep.ndims == 2) & (df_sweep.num_frames == 2)]

fig, ax_dict = plt.subplot_mosaic(
    [["mi", "theta"], ["mi", "omega"]],
    figsize=(11, 7), dpi=200,
    gridspec_kw=dict(width_ratios=[1.3, 1], hspace=0.65, wspace=0.38),
)
for spec in PANEL_SPECS:
    plot_sweep_panel(ax_dict[spec['key']], d, spec, 'samples_n')
    ax_dict[spec['key']].set_xscale('log')

ax_dict['mi'].set_ylim([0, 10.2])
xx = np.arange(32, 1000, 10)
ax_dict['mi'].plot(xx, np.log2(xx), c='k', ls='--', alpha=0.6)

for key, ax in ax_dict.items():
    ax.set_xticks([2, 4, 8, 16, 32, 64, 128, 256, 512, 1000])
    ax.set_xticklabels(['2', '4', '8', '16', '32', '64', '128', '256', '512', '1000'])
    ax.set_xlabel('Training videos N', fontsize=FS_AXIS_LABEL)
    ax.tick_params(labelsize=FS_TICK)
    ax.grid(alpha=GRID_ALPHA, linewidth=0.5)
    spec = next(s for s in PANEL_SPECS if s['key'] == key)
    ax.set_ylabel(spec['ylabel'], fontsize=FS_AXIS_LABEL)
    ax.set_title(spec['title'], fontsize=FS_TITLE)

for ax, lbl in zip([ax_dict['mi'], ax_dict['theta'], ax_dict['omega']], 'ABC'):
    y = 1.05 if lbl == 'A' else 1.12
    ax.text(-0.05, y, lbl, transform=ax.transAxes, fontsize=FS_PANEL_LABEL)

plt.savefig(SAVE_DIR / 'Fig_Sweep_vs_N.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## Figure: Pendulum phase portrait
θ–ω Cartesian (panel A), polar (panel B), learned latent space Z₁–Z₂ (panel C).

In [ ]:
# ── font sizes & cosmetics ────────────────────────────────────────────────
FS_TICK_LABEL, FS_AXIS_LABEL, FS_TITLE = 15, 18, 20
FS_PANEL_LABEL, FS_POLAR_RADIAL = 22, 13
GRID_ALPHA = 0.10
N_SHOW, RNG_SEED = 1200, 0  # show all trajectories
N_YTICKS = 5

pp = np.load(FIGDATA / 'phase_portrait.npz', allow_pickle=False)
theta_wrapped_deg = pp['theta_wrapped_deg']      # (1200, 60)
omega_rad         = pp['omega_rad_per_s']        # (1200, 60)
omega_plot        = omega_rad * _OMEGA_SCALE     # deg/s
omega_label       = r'$\omega$ ($^\circ$/s)'
zx_latent         = pp['zx_latent']              # (1200, numD, 2)
numD              = zx_latent.shape[1]
N_traj            = theta_wrapped_deg.shape[0]

rng        = np.random.default_rng(RNG_SEED)
sel_idx    = rng.choice(N_traj, size=min(N_SHOW, N_traj), replace=False)
traj_colors = plt.get_cmap('tab10')(rng.uniform(0, 1, len(sel_idx)))
theta_wrapped_rad = np.deg2rad(theta_wrapped_deg)

_lw, _alpha = 3.0, 1.0
fig = plt.figure(figsize=(21, 6), dpi=200)
ax_cart  = fig.add_subplot(1, 3, 1)
ax_polar = fig.add_subplot(1, 3, 2, projection='polar')
ax_lat   = fig.add_subplot(1, 3, 3)

for j, i in enumerate(sel_idx):
    c = traj_colors[j]
    _wrap_mask = np.concatenate([[False], np.abs(np.diff(theta_wrapped_deg[i])) > 180])
    _th = np.ma.array(theta_wrapped_deg[i], mask=_wrap_mask)
    _om = np.ma.array(omega_plot[i],        mask=_wrap_mask)
    ax_cart.plot(_th, _om, color=c, lw=_lw, alpha=_alpha)
    ax_polar.plot(theta_wrapped_rad[i], omega_plot[i], color=c, lw=_lw, alpha=_alpha)
    ax_lat.plot(zx_latent[i, :, 0], zx_latent[i, :, 1], color=c, lw=_lw, alpha=_alpha)

# Cartesian cosmetics
_tick_degs = [-180, -135, -90, -45, 0, 45, 90, 135, 180]
ax_cart.set_xlim(-180, 180)
ax_cart.set_xticks(_tick_degs)
ax_cart.set_xticklabels([rf'${d}^\circ$' for d in _tick_degs])
ax_cart.tick_params(labelsize=FS_TICK_LABEL)
ax_cart.yaxis.set_major_locator(plt.MaxNLocator(N_YTICKS))
ax_cart.axhline(0, color='k', lw=0.5, ls='--', alpha=0.3)
ax_cart.axvline(0, color='k', lw=0.5, ls='--', alpha=0.3)
ax_cart.set_xlabel(r'$\theta$ ($^\circ$)', fontsize=FS_AXIS_LABEL)
ax_cart.set_ylabel(omega_label, fontsize=FS_AXIS_LABEL)
ax_cart.grid(alpha=GRID_ALPHA, linewidth=0.5)

# Polar cosmetics
ax_polar.set_theta_zero_location('E')
_theta_grid_deg    = [0, 45, 90, 135, 180, 225, 270, 315]
_theta_grid_labels = [rf'${(d if d <= 180 else d - 360):.0f}^\circ$' for d in _theta_grid_deg]
ax_polar.set_thetagrids(_theta_grid_deg, labels=_theta_grid_labels, fontsize=FS_TICK_LABEL)
ax_polar.yaxis.set_major_locator(plt.MaxNLocator(N_YTICKS))
ax_polar.set_yticklabels([])
ax_polar.grid(alpha=GRID_ALPHA, linewidth=0.5)

# Latent cosmetics
ax_lat.set_xlabel(r'$Z_1$', fontsize=FS_AXIS_LABEL)
ax_lat.set_ylabel(r'$Z_2$', fontsize=FS_AXIS_LABEL)
ax_lat.tick_params(labelsize=FS_TICK_LABEL)
ax_lat.grid(alpha=GRID_ALPHA, linewidth=0.5)

fig.tight_layout()
fig.canvas.draw()
# polar radial annotations
_r_ticks = [r for r in ax_polar.get_yticks()]
_fmt = lambda r: f'{r:.0f}' if r == int(r) else f'{r:.1f}'
for r in _r_ticks:
    ax_polar.annotate(_fmt(r), xy=(np.pi/2 - np.pi/8, r), xycoords='data',
                      ha='center', va='bottom', fontsize=FS_POLAR_RADIAL, zorder=20)

# panel titles
_TITLES = [
    (ax_cart,  'Experimental Data: Phase space'),
    (ax_polar, 'Experimental Data: Polar coordinates'),
    (ax_lat,   'Learned latent space'),
]
_title_y = ax_cart.get_position().ymax + 0.07
for ax, txt in _TITLES:
    _xc = (ax.get_position().xmin + ax.get_position().xmax) / 2
    fig.text(_xc, _title_y, txt, ha='center', va='bottom', fontsize=FS_TITLE,
             transform=fig.transFigure)

for ax, lbl in zip([ax_cart, ax_polar, ax_lat], 'ABC'):
    ax.text(-0.05, 1.05, lbl, transform=ax.transAxes, fontsize=FS_PANEL_LABEL)

plt.savefig(SAVE_DIR / 'Fig_Phase_Space_Traj_Exp_Learned.png',
            dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## Figure: Learned embedding colored by θ and ω (kz=2, N=1000)

In [ ]:
FS_TICK, FS_AXIS_LABEL, FS_TITLE, FS_PANEL_LABEL, FS_CBAR = 12, 14, 15, 20, 12
GRID_ALPHA = 0.10
SUBSAMPLE = 100_000

em = np.load(FIGDATA / 'embedding_kz2.npz', allow_pickle=False)
Z         = em['Z']
theta_deg = em['theta_deg']
omega_dps = em['omega_dps']

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

_panels = [
    ('theta', theta_deg, CMAP_THETA, r'$\theta$ ($^\circ$)', (-180, 180), [-180, -90, 0, 90, 180]),
    ('omega', omega_dps, CMAP_OMEGA, r'$\omega$ ($^\circ$/s)', None, None),
]
for ax, (key, vals, cmap, label, vlim, ticks) in zip(axes, _panels):
    if vlim is None:
        lim = np.percentile(np.abs(vals), 99)
        vmin, vmax = -lim, lim
    else:
        vmin, vmax = vlim
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=vals, s=4, alpha=0.35, linewidths=0,
                    cmap=cmap, vmin=vmin, vmax=vmax, rasterized=True)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(label, fontsize=FS_CBAR)
    cbar.ax.tick_params(labelsize=FS_TICK)
    if ticks is not None: cbar.set_ticks(ticks)
    ax.set_xlabel(r'$Z_1$', fontsize=FS_AXIS_LABEL)
    ax.set_ylabel(r'$Z_2$', fontsize=FS_AXIS_LABEL)
    ax.set_title(label, fontsize=FS_TITLE)
    ax.tick_params(labelsize=FS_TICK)
    ax.grid(alpha=GRID_ALPHA, linewidth=0.5)

for ax, lbl in zip(axes, 'AB'):
    ax.text(-0.05, 1.05, lbl, transform=ax.transAxes, fontsize=FS_PANEL_LABEL)

fig.tight_layout()
plt.savefig(SAVE_DIR / 'Fig_Angle_Omega_Colored_Embeddings.png',
            dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## Figure: Decoder flow + rollout RMSE (test set)

Left: decoder flow field (red quiver), background scatter of test embeddings, three example trajectories (solid = true, dashed = predicted mean from 40 stochastic particles).  
Right: rollout RMSE vs frame (top = latent, bottom = wrapped θ in degrees; shaded band = ±1 SEM across 200 test trials).

*To change which trajectories are displayed:* re-run `python precompute_figure_data.py --only rollout_test --force` after editing `vis_seed` in `task_rollout_test`.

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from matplotlib.lines import Line2D

FS_TICK, FS_AXIS_LABEL, FS_PANEL_LABEL, FS_LEGEND = 12, 14, 20, 9
GRID_ALPHA = 0.10

r = np.load(FIGDATA / 'rollout_test.npz', allow_pickle=False)
x_frames         = r['x_frames']
latent_rmse_mean = r['latent_rmse_mean']
latent_rmse_sem  = r['latent_rmse_sem']
theta_err_mean   = r['theta_err_mean']
theta_err_sem    = r['theta_err_sem']
quiver_xy        = r['quiver_xy']
quiver_uv        = r['quiver_uv']
fix_up           = r['fix_up']
fix_dn           = r['fix_dn']
test_z_bg        = r['test_z_bg']
vis_true_z       = r['vis_true_z']    # (n_vis, numD, 2)
vis_pred_z       = r['vis_pred_z']    # (n_rollout+1, n_vis, 2)
_ro_r0           = r['r0']
_ro_r1           = r['r1']
nf               = int(r['num_frames'])
n_rollout        = int(r['n_rollout'])

fig = plt.figure(figsize=(14, 7), dpi=150)
gs  = gridspec.GridSpec(2, 2, figure=fig,
                        width_ratios=[1.3, 1], hspace=0.45, wspace=0.38)
ax_quiv  = fig.add_subplot(gs[:, 0])
ax_lat   = fig.add_subplot(gs[0, 1])
ax_theta = fig.add_subplot(gs[1, 1])

# ── quiver panel ──────────────────────────────────────────────────────────
ax_quiv.scatter(test_z_bg[:, 0], test_z_bg[:, 1], s=1.5, alpha=0.06,
                c='lightgray', linewidths=0, rasterized=True)
ax_quiv.quiver(quiver_xy[:, 0], quiver_xy[:, 1],
               quiver_uv[:, 0], quiver_uv[:, 1],
               color='r', alpha=0.20, scale=40)

if not np.isnan(fix_up).any():
    ax_quiv.plot(fix_up[0], fix_up[1], 'r*', ms=15, zorder=5,
                 label=r'$\theta \approx \pi$ (unstable)')
if not np.isnan(fix_dn).any():
    ax_quiv.plot(fix_dn[0], fix_dn[1], 'y*', ms=15, zorder=5,
                 label=r'$\theta \approx 0$ (stable)')

n_vis = vis_true_z.shape[0]
NUM_SAMPLES = 40
_sem_scale = 1.0 / np.sqrt(NUM_SAMPLES)
_VIS_COLORS = [f'C{i}' for i in range(n_vis)]

for j, c in enumerate(_VIS_COLORS):
    tz_full = vis_true_z[j]                          # (numD, 2)
    tz_sub  = vis_true_z[j, ::nf]                    # windows visited by rollout
    pz      = vis_pred_z[:, j, :]                    # (n_rollout+1, 2)

    ax_quiv.plot(tz_full[:, 0], tz_full[:, 1], '-', color=c, lw=1.2, alpha=0.9)
    ax_quiv.scatter(tz_sub[:, 0], tz_sub[:, 1], color=c, s=30, zorder=4, linewidths=0)
    ax_quiv.plot(pz[:, 0], pz[:, 1], '--', color=c, lw=1.2, alpha=0.9)
    ax_quiv.scatter(pz[:, 0], pz[:, 1], marker='s', color=c, s=30,
                    alpha=1.0, zorder=5, linewidths=0)
    ax_quiv.scatter(tz_full[0, 0], tz_full[0, 1], marker='^', facecolor='w',
                    s=80, zorder=6, linewidths=2, ec=c)

_fp_handles, _ = ax_quiv.get_legend_handles_labels()
_leg_handles = [
    Line2D([0], [0], color='k', ls='-',    marker='o', ms=4, label='True'),
    Line2D([0], [0], color='k', ls='--',   marker='s', ms=4, label='Predicted mean'),
    Line2D([0], [0], color='k', ls='None', marker='^', ms=8,
           markerfacecolor='w', markeredgecolor='k', markeredgewidth=2,
           label=r'Start ($t=0$)'),
]
ax_quiv.legend(handles=_fp_handles + _leg_handles,
               fontsize=FS_LEGEND, loc='upper right')

ax_quiv.set_xlim(_ro_r0); ax_quiv.set_ylim(_ro_r1)
ax_quiv.set_xlabel(r'$Z_1$', fontsize=FS_AXIS_LABEL)
ax_quiv.set_ylabel(r'$Z_2$', fontsize=FS_AXIS_LABEL)
ax_quiv.grid(alpha=GRID_ALPHA, linewidth=0.5)

# ── latent RMSE ───────────────────────────────────────────────────────────
_ln_lat, = ax_lat.plot(x_frames, latent_rmse_mean, lw=1.4)
_c_lat = _ln_lat.get_color()
ax_lat.fill_between(x_frames,
                    latent_rmse_mean - latent_rmse_sem,
                    latent_rmse_mean + latent_rmse_sem,
                    alpha=0.25, color=_c_lat)
ax_lat.scatter(x_frames, latent_rmse_mean, s=20, zorder=5, color=_c_lat)
ax_lat.set_xlabel('Frame', fontsize=FS_AXIS_LABEL)
ax_lat.set_ylabel('Latent Rollout RMSE', fontsize=FS_AXIS_LABEL)
ax_lat.grid(alpha=GRID_ALPHA, linewidth=0.5)

# ── θ RMSE ────────────────────────────────────────────────────────────────
_ln_th, = ax_theta.plot(x_frames, theta_err_mean, lw=1.4)
_c_th = _ln_th.get_color()
ax_theta.fill_between(x_frames,
                      theta_err_mean - theta_err_sem,
                      theta_err_mean + theta_err_sem,
                      alpha=0.25, color=_c_th)
ax_theta.scatter(x_frames, theta_err_mean, s=20, zorder=5, color=_c_th)
ax_theta.set_xlabel('Frame', fontsize=FS_AXIS_LABEL)
ax_theta.set_ylabel(r'$\theta$ Rollout RMSE ($^\circ$)', fontsize=FS_AXIS_LABEL)
ax_theta.grid(alpha=GRID_ALPHA, linewidth=0.5)

for _ax in [ax_quiv, ax_lat, ax_theta]:
    _ax.tick_params(labelsize=FS_TICK)

for ax, lbl in zip([ax_quiv, ax_lat, ax_theta], 'ABC'):
    y = 1.05 if lbl == 'A' else 1.12
    ax.text(-0.05, y, lbl, transform=ax.transAxes, fontsize=FS_PANEL_LABEL)

fig.tight_layout()
plt.savefig(SAVE_DIR / 'Fig_Long_Rollout_Test_Set.png',
            dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

---
# Supplementary Figures

These panels also read directly from the precomputed files already committed in `figure_data/`.
You only need to rerun `precompute_figure_data.py` if you want the supplementary figures to reflect new training outputs.


## Supp.: Sweep vs N at kz=8 + PCA/Intrinsic dimensionality

In [ ]:
from matplotlib.gridspec import GridSpec

# Temporarily switch to mean_std for error-bar + fill display
_saved_plot_mode = PLOT_MODE
PLOT_MODE = 'mean_std'

FS_TICK, FS_AXIS_LABEL, FS_TITLE, FS_PANEL_LABEL, FS_SUPTITLE = 12, 14, 15, 20, 20
FS_INSET_TICK, FS_INSET_LABEL, FS_MLE_INSET, FS_LEGEND, FS_CBAR = 9, 9, 9, 8, 12
GRID_ALPHA, INSET_GRID_ALPHA = 0.10, 0.10
MLE_INSET_LOG_Y = True
INSET_REF_PREFACTOR_THETA = 300
INSET_REF_PREFACTOR_OMEGA = 5000
kz, nF = 8, 2

# ── sweep data ────────────────────────────────────────────────────────────
d = df_sweep[(df_sweep.ndims == kz) & (df_sweep.num_frames == nF)]

# ── load PCA/ID precomputed ───────────────────────────────────────────────
pc = np.load(FIGDATA / 'pca_kz8.npz', allow_pickle=False)
sv_norm      = pc['sv_norm']
pr           = float(pc['pr'])
id_twonn     = float(pc['id_twonn'])
K_curve      = pc['K_curve']
id_mle_curve = pc['id_mle_curve']
z_pc         = pc['z_pc']
theta_deg_k8 = pc['theta_deg']
var_exp      = pc['var_exp']
dz8          = int(pc['dz'])

fig = plt.figure(figsize=(16, 11), dpi=200)
gs  = GridSpec(2, 6, figure=fig, hspace=0.38, wspace=0.55)
axes_top = [fig.add_subplot(gs[0, 0:2]), fig.add_subplot(gs[0, 2:4]), fig.add_subplot(gs[0, 4:6])]
ax_sv  = fig.add_subplot(gs[1, 0:3])
ax_emb = fig.add_subplot(gs[1, 3:6])

# Row 1: sweep panels with log-log insets
_inset_bounds = {'theta': (0.60, 0.55, 0.35, 0.40), 'omega': (0.60, 0.55, 0.35, 0.40)}
for ax, spec in zip(axes_top, PANEL_SPECS):
    plot_sweep_panel(ax, d, spec, 'samples_n', fill_std=True)
    ax.set_xscale('log')
    ax.set_xticks([2, 4, 8, 16, 32, 64, 128, 256, 512, 1000])
    ax.set_xticklabels(['2', '4', '8', '16', '32', '64', '128', '256', '512', '1000'])
    ax.set_xlabel('Training videos N', fontsize=FS_AXIS_LABEL)
    ax.set_ylabel(spec['ylabel'], fontsize=FS_AXIS_LABEL)
    ax.set_title(spec['title'], fontsize=FS_TITLE)
    ax.tick_params(labelsize=FS_TICK)
    ax.grid(alpha=GRID_ALPHA, linewidth=0.5)
    if spec['key'] in _inset_bounds:
        axin = ax.inset_axes(_inset_bounds[spec['key']])
        plot_sweep_panel(axin, d, spec, 'samples_n', fill_std=True)
        axin.set_xscale('log'); axin.set_yscale('log')
        axin.tick_params(labelsize=FS_INSET_TICK)
        axin.grid(alpha=INSET_GRID_ALPHA, linewidth=0.5)
        axin.set_xlabel('Training videos N', fontsize=FS_INSET_LABEL)
        axin.set_ylabel(spec['ylabel'], fontsize=FS_INSET_LABEL)
        pref = INSET_REF_PREFACTOR_THETA if spec['key'] == 'theta' else INSET_REF_PREFACTOR_OMEGA
        _N_ref = np.linspace(2, 1000, 200)
        axin.plot(_N_ref, pref / np.sqrt(_N_ref), 'k--', lw=0.8, alpha=0.6,
                  label=r'$\propto 1/\sqrt{N}$')

axes_top[0].set_ylim([0, 10.2])
xx = np.arange(2, 1000, 10)
axes_top[0].plot(xx, np.log2(xx), c='k', ls='--', alpha=0.6)

# Panel D: SV bar + MLE inset
ax_sv.bar(np.arange(1, dz8 + 1), sv_norm, color='steelblue', edgecolor='white', linewidth=0.6)
ax_sv.set_xlabel('Principal component', fontsize=FS_AXIS_LABEL)
ax_sv.set_ylabel(r'Singular value ($\sigma_i / \sigma_1$)', fontsize=FS_AXIS_LABEL)
ax_sv.set_xticks(np.arange(1, dz8 + 1))
ax_sv.set_title(f'Participation Ratio = {pr:.2f},  TwoNN = {id_twonn:.2f}', fontsize=FS_TITLE)
ax_sv.grid(axis='y', alpha=GRID_ALPHA, linewidth=0.6)
ax_sv.tick_params(labelsize=FS_TICK)

axin_mle = ax_sv.inset_axes([0.42, 0.42, 0.52, 0.48])
axin_mle.plot(K_curve, id_mle_curve, color='#e07b54', lw=1.4)
axin_mle.scatter(K_curve, id_mle_curve, color='#e07b54', s=12, zorder=5)
axin_mle.axhline(2, color='k', ls='--', lw=0.8, label='ID = 2')
axin_mle.set_xlabel('k (neighbours)', fontsize=FS_MLE_INSET)
axin_mle.set_ylabel('MLE ID', fontsize=FS_MLE_INSET)
axin_mle.tick_params(labelsize=FS_MLE_INSET)
axin_mle.set_xscale('log')
if MLE_INSET_LOG_Y: axin_mle.set_yscale('log')
axin_mle.grid(alpha=INSET_GRID_ALPHA, linewidth=0.5)
axin_mle.legend(fontsize=FS_LEGEND, loc='upper right')

# Panel E: PC1 vs PC2
SUBSAMPLE = 50_000
rng_sc = np.random.default_rng(0)
idx_sc = rng_sc.choice(len(z_pc), size=min(SUBSAMPLE, len(z_pc)), replace=False)
sc = ax_emb.scatter(z_pc[idx_sc, 0], z_pc[idx_sc, 1],
                    c=theta_deg_k8[idx_sc], s=4, alpha=0.35, linewidths=0,
                    cmap=CMAP_THETA, vmin=-180, vmax=180)
cbar = fig.colorbar(sc, ax=ax_emb, shrink=0.85)
cbar.set_label(r'$\theta$ ($^\circ$)', fontsize=FS_CBAR)
cbar.ax.tick_params(labelsize=FS_TICK)
cbar.set_ticks([-180, -90, 0, 90, 180])
ax_emb.set_xlabel(f'PC 1  ({var_exp[0]*100:.1f}% var)', fontsize=FS_AXIS_LABEL)
ax_emb.set_ylabel(f'PC 2  ({var_exp[1]*100:.1f}% var)', fontsize=FS_AXIS_LABEL)
ax_emb.grid(alpha=GRID_ALPHA, linewidth=0.5)
ax_emb.tick_params(labelsize=FS_TICK)

for ax, lbl in zip(axes_top + [ax_sv, ax_emb], 'ABCDE'):
    ax.text(-0.05, 1.05, lbl, transform=ax.transAxes, fontsize=FS_PANEL_LABEL)

fig.suptitle(rf'$k_z = {kz},\; n_F = {nF}$', fontsize=FS_SUPTITLE, y=0.98)
plt.savefig(SAVE_DIR / 'Fig_Supp_kz8_combined.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

# restore global mode
PLOT_MODE = _saved_plot_mode

## Supp.: Embedding vs N (kz=2, θ coloring)

In [ ]:
# ── font sizes & cosmetics (match pub notebook) ───────────────────────────
FS_TICK        = 12     # tick label size
FS_AXIS_LABEL  = 14     # axis label size
FS_TITLE       = 15     # panel title size
FS_CBAR_LABEL  = 12     # colorbar label size
FS_PANEL_LABEL = 20     # panel labels
GRID_ALPHA     = 0.10   # background grid transparency

evn = np.load(FIGDATA / 'embedding_vs_N.npz', allow_pickle=False)
Ns = [64, 256, 1000]

fig, axes = plt.subplots(1, len(Ns), figsize=(5 * len(Ns), 5), dpi=300)
for ax, N in zip(axes, Ns):
    Z  = evn[f'Z_N{N}']
    th = evn[f'theta_deg_N{N}']
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=th, s=4, alpha=0.35, linewidths=0,
                    cmap=CMAP_THETA, vmin=-180, vmax=180, rasterized=True)
    ax.set_title(rf"N = {N}", fontsize=FS_TITLE)
    ax.set_xlabel(r"$Z_0$", fontsize=FS_AXIS_LABEL)
    ax.set_ylabel(r"$Z_1$", fontsize=FS_AXIS_LABEL)
    ax.tick_params(labelsize=FS_TICK)
    ax.grid(alpha=GRID_ALPHA, linewidth=0.5)

fig.tight_layout()

# shared horizontal colorbar below all panels (added AFTER tight_layout)
cbar = fig.colorbar(sc, ax=axes, orientation="horizontal",
                    fraction=0.05, pad=0.2, shrink=0.6)
cbar.set_label(r"$\theta$ ($^\circ$)", fontsize=FS_CBAR_LABEL)
cbar.ax.tick_params(labelsize=FS_TICK)
cbar.set_ticks([-180, -90, 0, 90, 180])
cbar.set_ticklabels([r"$-180$", r"$-90$", r"$0$", r"$90$", r"$180$"])

for ax, s in zip(axes, ['A', 'B', 'C']):
    ax.text(-0.05, 1.05, s, transform=ax.transAxes, fontsize=FS_PANEL_LABEL)

plt.savefig(SAVE_DIR / 'Fig_Supp_Embedding_vs_N.png',
            dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## Supp.: Cylindrical projection (central-projection visualisation)

In [ ]:
# ── Cylindrical phase-space + central projection ─────────────────────────
# Verbatim port from paper_figures_pub.ipynb — only data loading at the top
# differs.  Every trajectory selection, seed, and cosmetic matches the pub
# notebook.

from mpl_toolkits.mplot3d import Axes3D                    # noqa: F401
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors

# ── load precomputed phys arrays ──────────────────────────────────────────
_cy = np.load(FIGDATA / 'cylindrical_phys.npz', allow_pickle=False)
theta_wrapped   = np.deg2rad(_cy['theta_wrapped_deg'])    # (1200, 60) rad, wrapped (−π, π]
omega_plot      = _cy['omega_dps']                        # (1200, 60) deg/s
omega_label     = r"$\omega$ ($^\circ$/s)"
N_traj          = theta_wrapped.shape[0]

# ── adjustable parameters ─────────────────────────────────────────────────
N_TRAJ_CYL          = 15     # number of trajectories to show
RAY_STRIDE          = 60     # one projection ray every RAY_STRIDE time-steps
CYL_R               = 1.0    # cylinder radius
CYL_ALPHA           = 0.10   # cylinder surface transparency
H_PROJ_FACTOR       = 2.0    # H_proj = omega_max × this  (above cylinder top)
Z_DISK_FACTOR       = 2.0    # z_disk = −omega_max × this (below cylinder base)
DISK_ALPHA          = 0.0    # disk fill transparency (1 = solid)
SHOW_DISK_OUTLINE   = True
SHOW_Z_AXIS         = True
RAY_ALPHA           = 0.4
TRAJ_LW             = 1.8
TRAJ_ALPHA_CYL      = 0.8
TRAJ_ALPHA_DISK     = 1.0
RNG_SEED_CYL        = 9
CYL_TRAJ_INDICES    = [1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 19, 31, 34, 35]
ELEV                = 25
AZIM                = -55

# ── font sizes ────────────────────────────────────────────────────────────
FS_THETA_LABELS     = 11
FS_OMEGA_LABELS     = 11
FS_ZLABEL           = 12
FS_TITLE            = 13

# ── disk polar annotations ────────────────────────────────────────────────
SHOW_THETA_LABELS   = True
THETA_LABEL_R_FACTOR = 1.30
SHOW_OMEGA_LABELS   = True
OMEGA_LABEL_DIR_DEG = 200
OMEGA_LABEL_ROT     = 100
OMEGA_TICK_VALS     = []
N_OMEGA_TICKS       = 4
CYL_OMEGA_MAX       = 1000
Z_TICK_VALS         = [-1000, -500, 0, 500, 1000]
DISK_TICK_DEGS      = [-135, -90, -45, 0, 45, 90, 135, 180]

# ── fixed points ──────────────────────────────────────────────────────────
SHOW_FIXED_POINTS   = True
FP_STABLE_COLOR     = "limegreen"
FP_UNSTABLE_COLOR   = "red"
FP_MARKERSIZE       = 180
FP_STAR_R_OUTER     = 0.25
FP_STAR_R_INNER     = 0.1
FP_SHADE_BLEND      = 0
FP_DISK_STAR_R_OUTER = 0.075
FP_DISK_STAR_R_INNER = 0.03
FP_CYL_MARKER_R_FRONT = 1.16
FP_CYL_MARKER_R_BACK  = 0.96
FP_CYL_STEM_LW        = 0.9
FP_BACK_ALPHA         = 0.65
LEGEND_BBOX          = (0.03, 0.98)
LAYOUT_RECT          = [0.01, 0.01, 0.99, 0.99]

# ── derive geometry from data ─────────────────────────────────────────────
omega_max  = float(CYL_OMEGA_MAX)
H_proj     = omega_max * H_PROJ_FACTOR
z_disk     = -omega_max * Z_DISK_FACTOR

r_proj_max = CYL_R * (H_proj - z_disk) / (H_proj - omega_max)
r_disk_vis = r_proj_max * 1.15

print(f"omega_max={omega_max:.1f}  H_proj={H_proj:.1f}  "
      f"z_disk={z_disk:.1f}  r_proj_max={r_proj_max:.2f}·R")

# ── select trajectories ───────────────────────────────────────────────────
if CYL_TRAJ_INDICES is not None:
    cyl_sel = np.array(CYL_TRAJ_INDICES)
else:
    rng_cyl = np.random.default_rng(RNG_SEED_CYL)
    cyl_sel = rng_cyl.choice(N_traj, size=N_TRAJ_CYL, replace=False)
print(f"Trajectories: {cyl_sel.tolist()}")
cyl_cols = [plt.cm.tab10((k % 10) / 10) for k in range(len(cyl_sel))]

# ── figure ────────────────────────────────────────────────────────────────
fig_cyl = plt.figure(figsize=(8, 10), dpi=150)
ax3d    = fig_cyl.add_subplot(111, projection="3d", computed_zorder=False)

# ── cylinder surface ──────────────────────────────────────────────────────
_th_s = np.linspace(-np.pi, np.pi, 120)
_TH, _ZS = np.meshgrid(_th_s, np.array([-omega_max, omega_max]))
ax3d.plot_surface(
    CYL_R * np.cos(_TH), CYL_R * np.sin(_TH), _ZS,
    alpha=CYL_ALPHA, color="steelblue", linewidth=0, antialiased=True, zorder=5,
)
for _z_rim in [omega_max, -omega_max]:
    ax3d.plot(CYL_R * np.cos(_th_s), CYL_R * np.sin(_th_s),
              np.full_like(_th_s, _z_rim),
              color="steelblue", lw=0.8, alpha=0.5)

# ── projection point S ────────────────────────────────────────────────────
ax3d.scatter([0], [0], [H_proj], color="black", s=60, zorder=10)

# ── disk below ────────────────────────────────────────────────────────────
_th_d = np.linspace(0, 2 * np.pi, 150)
_r_d  = np.linspace(0, r_disk_vis, 50)
_TD, _RD = np.meshgrid(_th_d, _r_d)
ax3d.plot_surface(
    _RD * np.cos(_TD), _RD * np.sin(_TD), np.full_like(_RD, z_disk),
    alpha=DISK_ALPHA, color="lightblue", linewidth=0, antialiased=True,
)
if SHOW_DISK_OUTLINE:
    ax3d.plot(r_disk_vis * np.cos(_th_d), r_disk_vis * np.sin(_th_d),
              np.full_like(_th_d, z_disk),
              color="steelblue", lw=0.8, alpha=0.6)

# ── disk polar annotations ────────────────────────────────────────────────
if SHOW_THETA_LABELS:
    for _deg in DISK_TICK_DEGS:
        _rad   = np.deg2rad(_deg)
        _r_lbl = r_disk_vis * THETA_LABEL_R_FACTOR
        ax3d.plot(
            [r_disk_vis * 0.93 * np.cos(_rad), r_disk_vis * np.cos(_rad)],
            [r_disk_vis * 0.93 * np.sin(_rad), r_disk_vis * np.sin(_rad)],
            [z_disk, z_disk], color="gray", lw=0.8,
        )
        ax3d.text(
            _r_lbl * np.cos(_rad), _r_lbl * np.sin(_rad), z_disk,
            rf"${_deg}^\circ$", ha="center", va="center",
            fontsize=FS_THETA_LABELS,
        )

if SHOW_OMEGA_LABELS:
    _dir_rad       = np.deg2rad(OMEGA_LABEL_DIR_DEG)
    _cos_d, _sin_d = np.cos(_dir_rad), np.sin(_dir_rad)
    if OMEGA_TICK_VALS is not None:
        _om_ticks = np.asarray(OMEGA_TICK_VALS, dtype=float)
    else:
        _om_ticks = np.linspace(0, omega_max, N_OMEGA_TICKS + 1)[1:]
    _tick_half = r_disk_vis * 0.025
    _perp_cos, _perp_sin = -_sin_d, _cos_d
    for _om in _om_ticks:
        _r = CYL_R * (H_proj - z_disk) / (H_proj - _om)
        ax3d.plot(
            [_r * _cos_d - _tick_half * _perp_cos,
             _r * _cos_d + _tick_half * _perp_cos],
            [_r * _sin_d - _tick_half * _perp_sin,
             _r * _sin_d + _tick_half * _perp_sin],
            [z_disk, z_disk], color="gray", lw=0.7,
        )
        ax3d.text(
            _r * _cos_d * 1.14, _r * _sin_d * 1.14, z_disk,
            f"{_om:.0f}", ha="center", va="center",
            fontsize=FS_OMEGA_LABELS, color="dimgray",
            rotation=OMEGA_LABEL_ROT,
        )
    ax3d.plot(
        [0, r_disk_vis * _cos_d], [0, r_disk_vis * _sin_d],
        [z_disk, z_disk], color="gray", lw=0.6, ls="-", alpha=0.5,
    )

# ── trajectories, projected trajectories, and sparse rays ─────────────────
for j, i in enumerate(cyl_sel):
    c    = cyl_cols[j]
    th_i = theta_wrapped[i]
    om_i = omega_plot[i]
    ok   = np.abs(om_i) <= omega_max

    x_c  = CYL_R * np.cos(th_i)
    y_c  = CYL_R * np.sin(th_i)
    z_c  = om_i
    t_p  = (z_disk - H_proj) / (z_c - H_proj)
    x_p  = t_p * x_c
    y_p  = t_p * y_c

    ax3d.plot(x_c[ok], y_c[ok], z_c[ok], color=c, lw=TRAJ_LW,
              alpha=TRAJ_ALPHA_CYL, zorder=4)
    ax3d.plot(x_p[ok], y_p[ok], np.full(ok.sum(), z_disk),
              color=c, lw=TRAJ_LW * 0.75, alpha=TRAJ_ALPHA_DISK, zorder=4)

    ray_pts = np.where(ok)[0][::RAY_STRIDE]
    for k in ray_pts:
        ax3d.plot(
            [0, x_p[k]], [0, y_p[k]], [H_proj, z_disk],
            color="dimgray", lw=0.55, ls="--", alpha=RAY_ALPHA, zorder=2,
        )

# ── fixed points ──────────────────────────────────────────────────────────
if SHOW_FIXED_POINTS:
    _fp_defs = [
        (0.0,    FP_STABLE_COLOR,   r"stable ($\theta =0, \omega =0$)"),
        (np.pi,  FP_UNSTABLE_COLOR, r"unstable ($\theta = \pi, \omega =0$)"),
    ]
    _t_fp_disk = (z_disk - H_proj) / (0.0 - H_proj)
    _azim_rad  = np.deg2rad(AZIM)

    def _flat_star_verts(xc, yc, z, r_out, r_in, n=5):
        n_verts = 2 * n
        ang0  = np.arctan2(yc, xc)
        ang   = np.linspace(0, 2*np.pi, n_verts, endpoint=False) + ang0 - np.pi/2
        radii = np.where(np.arange(n_verts) % 2 == 0, r_out, r_in)
        return list(zip(xc + radii*np.cos(ang), yc + radii*np.sin(ang),
                        np.full(n_verts, z)))

    _fp_handles = []

    for _th_fp, _c_fp, _fp_label in _fp_defs:
        _xc_fp = CYL_R * np.cos(_th_fp)
        _yc_fp = CYL_R * np.sin(_th_fp)
        _front = (np.cos(_th_fp)*np.cos(_azim_rad)
                  + np.sin(_th_fp)*np.sin(_azim_rad)) > 0
        _marker_r = FP_CYL_MARKER_R_FRONT if _front else FP_CYL_MARKER_R_BACK
        _marker_alpha = 1.0 if _front else FP_BACK_ALPHA
        _xd_fp = _t_fp_disk * _xc_fp
        _yd_fp = _t_fp_disk * _yc_fp
        _xm_fp = _marker_r * np.cos(_th_fp)
        _ym_fp = _marker_r * np.sin(_th_fp)
        _base  = np.array(mcolors.to_rgba(_c_fp))
        _gray  = np.array([0.65, 0.65, 0.65, 1.0])
        _cyl_rgba = _base if _front else (1-FP_SHADE_BLEND)*_base + FP_SHADE_BLEND*_gray

        if _front:
            ax3d.plot([_xc_fp, _xm_fp], [_yc_fp, _ym_fp], [0.0, 0.0],
                      color=_cyl_rgba, lw=FP_CYL_STEM_LW, alpha=0.95, zorder=14)
        ax3d.scatter([_xm_fp], [_ym_fp], [0.0],
                     color=[_cyl_rgba], marker="*", s=FP_MARKERSIZE,
                     edgecolors="k", linewidths=0.6,
                     depthshade=False, alpha=_marker_alpha,
                     zorder=15 if _front else 1)
        _fp_handles.append(
            Line2D([0], [0], linestyle="None", marker="*", markersize=11,
                   markerfacecolor=_base, markeredgecolor="k", markeredgewidth=0.6,
                   label=_fp_label)
        )

        _dsverts = _flat_star_verts(_xd_fp, _yd_fp, z_disk,
                                    FP_DISK_STAR_R_OUTER * r_disk_vis,
                                    FP_DISK_STAR_R_INNER * r_disk_vis)
        ax3d.add_collection3d(Poly3DCollection([_dsverts],
                               facecolor=_base, edgecolor="k",
                               linewidths=0.4, zorder=10))

        ax3d.plot([0, _xd_fp], [0, _yd_fp], [H_proj, z_disk],
                  color="dimgray", lw=0.7, ls="--", alpha=RAY_ALPHA, zorder=2)

    _leg = ax3d.legend(handles=_fp_handles, loc="upper left",
                      bbox_to_anchor=LEGEND_BBOX, borderaxespad=0.0,
                      framealpha=0.7, fontsize=FS_THETA_LABELS)
    _leg.set_in_layout(False)

# ── cosmetics ─────────────────────────────────────────────────────────────
ax3d.set_xlabel("")
ax3d.set_ylabel("")
ax3d.set_xticks([])
ax3d.set_yticks([])
if SHOW_Z_AXIS:
    ax3d.set_zlabel(omega_label, labelpad=6, fontsize=FS_ZLABEL)
    if Z_TICK_VALS is not None:
        ax3d.set_zticks(Z_TICK_VALS)
else:
    ax3d.set_zlabel("")
    ax3d.set_zticks([])
_xy_lim = r_disk_vis * 1.02
ax3d.set_xlim(-_xy_lim, _xy_lim)
ax3d.set_ylim(-_xy_lim, _xy_lim)
ax3d.set_zlim(z_disk, 0.75*H_proj)
ax3d.set_box_aspect([1, 1, 1.2])
ax3d.view_init(elev=ELEV, azim=AZIM)

ax3d.xaxis.pane.fill = False
ax3d.yaxis.pane.fill = False
ax3d.zaxis.pane.fill = False
ax3d.xaxis.pane.set_edgecolor("white")
ax3d.yaxis.pane.set_edgecolor("white")
ax3d.zaxis.pane.set_edgecolor("white")
ax3d.grid(False)

fig_cyl.subplots_adjust(*LAYOUT_RECT)
plt.savefig(SAVE_DIR / 'Fig_Cylindrical_To_Polar_Projection.png',
            dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## Supp.: 5-panel embedding (θ, ω, KE, PE, Total energy)

In [ ]:
FS_TICK, FS_AXIS_LABEL, FS_TITLE, FS_PANEL_LABEL, FS_CBAR = 12, 14, 15, 20, 12
GRID_ALPHA = 0.10

em = np.load(FIGDATA / 'embedding_kz2.npz', allow_pickle=False)
Z = em['Z']

_PANELS = [
    ('theta',    em['theta_deg'],  CMAP_THETA,  r'$\theta$ ($^\circ$)',       (-180, 180), [-180, -90, 0, 90, 180]),
    ('omega',    em['omega_dps'],  CMAP_OMEGA,  r'$\omega$ ($^\circ$/s)',     None,        None),
    ('kinetic',  em['kinetic'],    CMAP_KE,     'Kinetic energy (J)',         None,        None),
    ('pot',      em['pot'],        CMAP_PE,     'Potential energy (J)',       'sym',       None),
    ('total',    em['total'],      CMAP_TE,     'Total energy (J)',           None,        None),
]

fig, axes = plt.subplots(1, 5, figsize=(25, 5), dpi=150)
for ax, (key, vals, cmap, label, vlim, ticks) in zip(axes, _PANELS):
    if vlim == 'sym':
        lim = np.abs(vals).max()
        vmin, vmax = -lim, lim
    elif vlim is None:
        lim = np.percentile(np.abs(vals), 99) if key == 'omega' else None
        vmin, vmax = (-lim, lim) if lim is not None else (vals.min(), vals.max())
    else:
        vmin, vmax = vlim
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=vals, s=4, alpha=0.35, linewidths=0,
                    cmap=cmap, vmin=vmin, vmax=vmax, rasterized=True)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(label, fontsize=FS_CBAR)
    cbar.ax.tick_params(labelsize=FS_TICK)
    if ticks is not None: cbar.set_ticks(ticks)
    ax.set_xlabel(r'$Z_1$', fontsize=FS_AXIS_LABEL)
    ax.set_ylabel(r'$Z_2$', fontsize=FS_AXIS_LABEL)
    ax.set_title(label, fontsize=FS_TITLE)
    ax.tick_params(labelsize=FS_TICK)
    ax.grid(alpha=GRID_ALPHA, linewidth=0.5)

for ax, lbl in zip(axes, 'ABCDE'):
    ax.text(-0.05, 1.05, lbl, transform=ax.transAxes, fontsize=FS_PANEL_LABEL)

fig.tight_layout()
plt.savefig(SAVE_DIR / 'Fig_Angle_Omega_Energies_Colored_Embeddings.png',
            dpi=SAVE_DPI, bbox_inches='tight')
plt.show()